# spi_nb_silver_construction

**Purpose.** Silver transformation of the MITMA tender tables for the
Construction source (source 4). Parses the raw XLS grid (ADR-013): removes the
per-file offset, keeps the monthly rows, forward-fills the year, casts the
values, extracts the Nacional, Cataluña and Madrid columns by position, and
builds `indicator_category` from the table (ADR-016). Checks the result
against the totals and annual sums the files carry, and raises on a mismatch.

**Inputs.** Delta table `spi_bronze_construction_raw` in the default
Lakehouse: positional columns `col_01` … `col_14` plus lineage, all string,
null where a file has fewer columns.

**Outputs.** Delta table `spi_silver_construction_cleaned` in the default
Lakehouse, written with `mode="overwrite"`. Columns: `region_name`,
`indicator_category`, `year`, `month`, `indicator_value`,
`_ingestion_timestamp`.

**Dependencies.** `pandas`, `deltalake`. Runs on the Fabric **Python** runtime
(no Spark session), per ADR-012 (2026-10-04 addendum).

**Exit contract.** On success, calls `notebookutils.notebook.exit()` with
`{"rows_written": <int>}`. On failure, the exception propagates uncaught, per
the 2026-09-30 addendum to ADR-009.

**Last updated.** 2026-10-06.

In [ ]:
import json
import logging
import re

import pandas as pd
from deltalake import DeltaTable, write_deltalake

# ────────────────────────────────────────────────────────────────────────────
# Tables — OneLake abfss:// URL, not /lakehouse/default/ (lessons-learned.md,
# Python notebooks)
# ────────────────────────────────────────────────────────────────────────────

TABLES_URI = (
    "abfss://bbffcd8e-0f61-436d-bee8-eac2d26d38b2@onelake.dfs.fabric.microsoft.com/"
    "c6048c3c-087e-4436-80fa-6013d1a04aeb/Tables"
)
SOURCE_TABLE = f"{TABLES_URI}/spi_bronze_construction_raw"
TARGET_TABLE = f"{TABLES_URI}/spi_silver_construction_cleaned"

# ────────────────────────────────────────────────────────────────────────────
# Layout constants (ADR-013 2026-10-06 addendum, ADR-016)
# ────────────────────────────────────────────────────────────────────────────

# Two tables, each split into a main and a continuation file. col_offset is
# the number of leading columns to remove so that all four files share one
# layout. 01401600.XLS also has one extra empty row; selecting rows by month
# label (below) makes that need no code.
FILES = {
    "01401400.XLS": {"table": 8, "part": "main", "col_offset": 0},
    "01401600.XLS": {"table": 8, "part": "cont", "col_offset": 1},
    "01402600.XLS": {"table": 9, "part": "main", "col_offset": 0},
    "01402800.XLS": {"table": 9, "part": "cont", "col_offset": 0},
}

INDICATOR_CATEGORY = {
    8: "Total construcción - Estado y Seguridad Social",
    9: "Total construcción - Entes Territoriales",
}

YEAR_COLUMN = "col_01"
MONTH_COLUMN = "col_02"
TOTAL_COLUMN = "col_03"  # main files only; includes NO REGIONAL

# Value columns after the offset. Main: TOTAL + 9 regions. Continuation: 11
# regions. The 20 region columns of a table sum to its TOTAL.
VALUE_COLUMNS = {
    "main": [f"col_{i:02d}" for i in range(3, 13)],
    "cont": [f"col_{i:02d}" for i in range(3, 14)],
}

# Column -> spi_dim_region.region_name, by position after the offset. Verified
# against the MITMA header image on 2026-10-06 (ADR-016 Decision 1). Values
# copied from spi_df_gold_dimensions, not retyped (ADR-010 §4).
REGION_COLUMNS = {
    "main": {"col_03": "Nacional", "col_12": "Cataluña"},
    "cont": {"col_06": "Madrid"},
}

MONTHS = {
    "ene.": 1, "feb.": 2, "mar.": 3, "abr.": 4, "may.": 5, "jun.": 6,
    "jul.": 7, "ago.": 8, "sep.": 9, "oct.": 10, "nov.": 11, "dic.": 12,
}

NO_TENDERS = "-"  # no tenders in the month = 0 (ADR-016 Decision 2)

TOLERANCE = 0.5  # thousand EUR; the files store unrounded values

OUTPUT_COLUMNS = [
    "region_name",
    "indicator_category",
    "year",
    "month",
    "indicator_value",
    "_ingestion_timestamp",
]

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Configure Logging
# ────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
log = logging.getLogger("silver_construction")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function storage_options and read_bronze
# ────────────────────────────────────────────────────────────────────────────

def storage_options() -> dict:
    """
    Build the OneLake storage options for deltalake.
    Returns:
        dict: Bearer token and Fabric endpoint flag.
    """
    return {
        "bearer_token": notebookutils.credentials.getToken("storage"),
        "use_fabric_endpoint": "true",
    }


def read_bronze() -> pd.DataFrame:
    """
    Read spi_bronze_construction_raw into pandas.
    Returns:
        pd.DataFrame: The Bronze table, all columns string.
    """
    df = DeltaTable(SOURCE_TABLE, storage_options=storage_options()).to_pandas()
    log.info(f"Read spi_bronze_construction_raw: {len(df)} rows")
    return df

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create functions split_files, to_values, monthly_rows and annual_rows
# ────────────────────────────────────────────────────────────────────────────

def split_files(bronze: pd.DataFrame) -> dict:
    """
    Split Bronze into one grid per file and remove each file's column offset.
    Row order is the Bronze read order (see monthly_rows).
    Args:
        bronze (pd.DataFrame): The Bronze table.
    Returns:
        dict: file_id -> grid with columns col_01 … and _ingestion_timestamp.
    Raises:
        ValueError: On an unknown _source_file, or a file of FILES missing.
    """
    found = set(bronze["_source_file"])
    unknown = sorted(found - set(FILES))
    if unknown:
        raise ValueError(f"Unknown _source_file values: {unknown}")
    missing = sorted(set(FILES) - found)
    if missing:
        raise ValueError(f"Files missing from Bronze: {missing}")

    positional = sorted(c for c in bronze.columns if re.fullmatch(r"col_\d{2}", c))
    grids = {}
    for file_id, spec in FILES.items():
        rows = bronze[bronze["_source_file"] == file_id]
        kept = positional[spec["col_offset"]:]
        grid = rows[kept].copy()
        grid.columns = positional[:len(kept)]
        grid["_ingestion_timestamp"] = rows["_ingestion_timestamp"]
        grids[file_id] = grid.reset_index(drop=True)
    return grids


def to_values(grid: pd.DataFrame, columns: list, file_id: str) -> pd.DataFrame:
    """
    Cast value cells to float64. '-' is 0 (ADR-016 Decision 2).
    Args:
        grid (pd.DataFrame): Rows of one file.
        columns (list): Value columns to cast.
        file_id (str): For error messages.
    Returns:
        pd.DataFrame: The columns as float64.
    Raises:
        ValueError: On a blank or null cell. Any other non-numeric string
        raises from pd.to_numeric.
    """
    cells = grid[columns]
    blank = cells.isna() | (cells == "")
    if blank.any().any():
        where = blank.stack()
        raise ValueError(f"{file_id}: blank value cells at {where[where].index.tolist()[:10]}")
    return cells.replace(NO_TENDERS, "0").apply(pd.to_numeric, errors="raise").astype("float64")


def monthly_rows(grid: pd.DataFrame, file_id: str) -> pd.DataFrame:
    """
    Keep the rows with a month label, forward-fill the year, and check that
    the months run backwards with no gap.
    Args:
        grid (pd.DataFrame): Grid of one file, offset removed.
        file_id (str): For error messages.
    Returns:
        pd.DataFrame: Monthly rows with int year and month, in file order.
    Raises:
        ValueError: On no monthly rows, a first month row without a year, a
        non-numeric year, or a break in the month sequence.
    """
    # Monthly rows only (ADR-016 Decision 3). Annual, V.* and footnote rows
    # have no month label.
    df = grid[grid[MONTH_COLUMN].isin(MONTHS)].copy()
    if df.empty:
        raise ValueError(f"{file_id}: no monthly rows")

    # The year sits on the first month row of each block (merged cell,
    # ADR-013 2026-10-06 addendum)
    year = df[YEAR_COLUMN].where(df[YEAR_COLUMN] != "").ffill()
    if year.isna().any():
        raise ValueError(f"{file_id}: first month row has no year")
    bad_years = sorted(set(year[~year.str.fullmatch(r"\d{4}")]))
    if bad_years:
        raise ValueError(f"{file_id}: non-numeric year labels: {bad_years}")
    df["year"] = year.astype("int64")
    df["month"] = df[MONTH_COLUMN].map(MONTHS).astype("int64")

    # Bronze stores no row order, so the forward-fill depends on the read
    # order (ADR-016 Consequences). Newest first, one month per row.
    period = df["year"] * 12 + df["month"]
    if not (period.diff().dropna() == -1).all():
        raise ValueError(f"{file_id}: months do not run backwards one month at a time")
    log.info(
        f"{file_id}: {len(df)} monthly rows, "
        f"{df['year'].iloc[-1]}-{df['month'].iloc[-1]:02d} to {df['year'].iloc[0]}-{df['month'].iloc[0]:02d}"
    )
    return df


def annual_rows(grid: pd.DataFrame) -> pd.DataFrame:
    """
    Select the annual total rows: a 4-digit year and no month label.
    Args:
        grid (pd.DataFrame): Grid of one file, offset removed.
    Returns:
        pd.DataFrame: Annual rows with an int year.
    """
    year = grid[YEAR_COLUMN].fillna("")
    month = grid[MONTH_COLUMN].fillna("")
    df = grid[year.str.fullmatch(r"\d{4}") & (month == "")].copy()
    df["year"] = df[YEAR_COLUMN].astype("int64")
    return df

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create functions check_totals and check_annual (ADR-016 Decision 5)
# ────────────────────────────────────────────────────────────────────────────

def check_totals(table: int, main: pd.DataFrame, cont: pd.DataFrame) -> None:
    """
    Guard 1: per (year, month), main TOTAL equals the sum of the 20 region
    columns across main and continuation.
    Args:
        table (int): 8 or 9, for error messages.
        main (pd.DataFrame): Monthly rows of the main file, values cast.
        cont (pd.DataFrame): Monthly rows of the continuation, values cast.
    Raises:
        ValueError: On a period present in only one file, or a mismatch
        above TOLERANCE.
    """
    keys = ["year", "month"]
    region_main = [c for c in VALUE_COLUMNS["main"] if c != TOTAL_COLUMN]
    left = main[keys + [TOTAL_COLUMN]].assign(regions_main=main[region_main].sum(axis=1))
    right = cont[keys].assign(regions_cont=cont[VALUE_COLUMNS["cont"]].sum(axis=1))
    joined = left.merge(right, on=keys, how="outer", indicator=True)

    unmatched = joined[joined["_merge"] != "both"]
    if not unmatched.empty:
        raise ValueError(f"Table {table}: periods in one file only: {unmatched[keys + ['_merge']].head(10).to_dict('records')}")

    diff = (joined[TOTAL_COLUMN] - joined["regions_main"] - joined["regions_cont"]).abs()
    bad = joined[diff > TOLERANCE]
    if not bad.empty:
        raise ValueError(f"Table {table}: TOTAL != sum of regions: {bad[keys].head(10).to_dict('records')}")
    log.info(f"Table {table}: totals identity holds for {len(joined)} months (max diff {diff.max():.6f})")


def check_annual(file_id: str, monthly: pd.DataFrame, annual: pd.DataFrame, columns: list) -> None:
    """
    Guard 2: for each annual row, the 12 monthly values of that year sum to it.
    Args:
        file_id (str): For error messages.
        monthly (pd.DataFrame): Monthly rows, values cast.
        annual (pd.DataFrame): Annual rows, values cast.
        columns (list): The kept columns to check.
    Raises:
        ValueError: On an annual year with other than 12 months, or a
        mismatch above TOLERANCE.
    """
    sums = monthly.groupby("year")[columns].sum()
    counts = monthly.groupby("year").size()
    for _, row in annual.iterrows():
        year = row["year"]
        if counts.get(year, 0) != 12:
            raise ValueError(f"{file_id}: annual row {year} has {counts.get(year, 0)} monthly rows, not 12")
        for col in columns:
            if abs(sums.at[year, col] - row[col]) > TOLERANCE:
                raise ValueError(
                    f"{file_id}: {year} {col}: months sum to {sums.at[year, col]}, annual row is {row[col]}"
                )
    log.info(f"{file_id}: annual check holds for {sorted(annual['year'].tolist())}")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create function transform
# ────────────────────────────────────────────────────────────────────────────

def transform(bronze: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the Silver rules to the Bronze rows (ADR-016).
    Args:
        bronze (pd.DataFrame): The Bronze table.
    Returns:
        pd.DataFrame: The Silver DataFrame.
    Raises:
        ValueError: From the parsing functions and guards, or on a duplicate
        (indicator_category, region_name, year, month).
    """
    grids = split_files(bronze)

    # 1. Monthly rows and annual rows per file, values cast
    monthly, annual = {}, {}
    for file_id, spec in FILES.items():
        columns = VALUE_COLUMNS[spec["part"]]
        m = monthly_rows(grids[file_id], file_id)
        m[columns] = to_values(m, columns, file_id)
        a = annual_rows(grids[file_id])
        a[columns] = to_values(a, columns, file_id)
        monthly[file_id], annual[file_id] = m, a

    # 2. Guard 1 per table: totals identity
    for table in sorted(INDICATOR_CATEGORY):
        parts = {spec["part"]: fid for fid, spec in FILES.items() if spec["table"] == table}
        check_totals(table, monthly[parts["main"]], monthly[parts["cont"]])

    # 3. Guard 2 per file, on the kept columns: annual check
    for file_id, spec in FILES.items():
        check_annual(file_id, monthly[file_id], annual[file_id], list(REGION_COLUMNS[spec["part"]]))

    # 4. Kept columns to long format, with region and indicator category
    frames = []
    for file_id, spec in FILES.items():
        for col, region_name in REGION_COLUMNS[spec["part"]].items():
            m = monthly[file_id]
            frames.append(pd.DataFrame({
                "region_name": region_name,
                "indicator_category": INDICATOR_CATEGORY[spec["table"]],
                "year": m["year"],
                "month": m["month"],
                "indicator_value": m[col],
                "_ingestion_timestamp": m["_ingestion_timestamp"],
            }))
    df = pd.concat(frames, ignore_index=True)

    # 5. (indicator_category, region_name, year, month) must be unique. Bronze
    #    is overwritten each run, so a duplicate is a defect: raise, do not
    #    deduplicate.
    keys = ["indicator_category", "region_name", "year", "month"]
    duplicated = df.duplicated(keys, keep=False)
    if duplicated.any():
        sample = df.loc[duplicated, keys].drop_duplicates().head(10).to_dict("records")
        raise ValueError(f"Duplicate (indicator_category, region_name, year, month): {sample}")

    df["_ingestion_timestamp"] = (
        pd.to_datetime(df["_ingestion_timestamp"], utc=True).astype("datetime64[us, UTC]")
    )

    return df[OUTPUT_COLUMNS]

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Create main to run the entire process
# ────────────────────────────────────────────────────────────────────────────

def main() -> dict:
    """
    Read Bronze, apply the Silver rules, and overwrite spi_silver_construction_cleaned.
    Returns:
        dict: rows_written.
    """
    bronze = read_bronze()
    silver = transform(bronze)
    write_deltalake(TARGET_TABLE, silver, mode="overwrite", storage_options=storage_options())
    log.info(f"Data stored in spi_silver_construction_cleaned: {len(silver)} rows")
    return {"rows_written": len(silver)}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# Exit contract — outside any try/except (ADR-009, 2026-09-30 addendum)
# ────────────────────────────────────────────────────────────────────────────
# Interactive runs: Fabric marks this cell with ✗ because exit() stops the
# notebook on purpose. Pipeline runs: the activity reports Succeeded and
# passes exitValue to the logging Script activity (ADR-009).

result = main()
notebookutils.notebook.exit(json.dumps(result))